### Healthcare Claims and Insurance Cost Analysis

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("datasets/healthcare_dataset_raw.csv")

In [4]:
df.head()

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


In [5]:
df.shape

(55500, 15)

In [6]:
df.columns

Index(['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition',
       'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider',
       'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date',
       'Medication', 'Test Results'],
      dtype='object')

In [7]:
df.describe(include="all")

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
count,55500,55500.000000,55500,55500,55500,55500,55500,55500,55500,55500.000000,55500.000000,55500,55500,55500,55500
unique,49992,NaN,2,8,6,1827,40341,39876,5,NaN,NaN,3,1856,5,3
top,DAvId muNoZ,NaN,Male,A-,Arthritis,2024-03-16,Michael Smith,LLC Smith,Cigna,NaN,NaN,Elective,2020-03-15,Lipitor,Abnormal
freq,3,NaN,27774,6969,9308,50,27,44,11249,NaN,NaN,18655,53,11140,18627
mean,NaN,51.539459,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25539.316097,301.134829,NaN,NaN,NaN,NaN
std,NaN,19.602454,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14211.454431,115.243069,NaN,NaN,NaN,NaN
min,NaN,13.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2008.492140,101.000000,NaN,NaN,NaN,NaN
25%,NaN,35.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13241.224652,202.000000,NaN,NaN,NaN,NaN
50%,NaN,52.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25538.069376,302.000000,NaN,NaN,NaN,NaN
75%,NaN,68.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37820.508436,401.000000,NaN,NaN,NaN,NaN


In [8]:
df.isnull().sum()

Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
dtype: int64

#### Data Cleaning

##### Creating a copy so that raw dataset stays the same

In [9]:
claims = df.copy()

##### Dates conversion

In [10]:
claims['Date of Admission'] = pd.to_datetime(claims['Date of Admission']) 
claims['Discharge Date'] = pd.to_datetime(claims['Discharge Date'])

In [11]:
claims[['Date of Admission', 'Discharge Date']].dtypes

Date of Admission    datetime64[ns]
Discharge Date       datetime64[ns]
dtype: object

##### checking duplicates

In [12]:
claims.duplicated().sum()

np.int64(534)

##### removing duplicates

In [13]:
claims = claims.drop_duplicates()

In [14]:
#verifying
claims.duplicated().sum()

np.int64(0)

##### Checking billing values

In [15]:
claims[claims['Billing Amount'] < 0].shape

(106, 15)

##### removing negative billing rows

In [16]:
claims = claims[claims['Billing Amount'] >= 0]

In [17]:
#verifying
claims['Billing Amount'].min()

np.float64(9.238787497393332)

In [18]:
claims.shape

(54860, 15)

#### Feature Engineering

##### claim ID

In [19]:
claims = claims.reset_index(drop=True) 
claims['Claim ID'] = ['CLM' + str(i).zfill(6) for i in range(1, len(claims)+1)]

##### Length of Stay

In [20]:
claims['Length of Stay'] = ( claims['Discharge Date'] - claims['Date of Admission'] ).dt.days

##### Age Groups

In [21]:
claims['Age Group'] = pd.cut( claims['Age'], bins=[0, 18, 35, 50, 65, 100], labels=['0-18', '19-35', '36-50', '51-65', '65+'] )

##### Cost Category

In [22]:
claims['Cost Category'] = pd.qcut( claims['Billing Amount'], q=3, labels=['Low', 'Medium', 'High'] )

##### claim status

In [23]:
np.random.seed(42) 
claims['Claim Status'] = np.random.choice( ['Approved', 'Denied', 'Pending'], size=len(claims), p=[0.78, 0.12, 0.10] )

In [24]:
#verifying
claims['Claim Status'].value_counts(normalize=True) * 100

Claim Status
Approved    78.155304
Denied      11.892089
Pending      9.952607
Name: proportion, dtype: float64

##### Reimbursement Amount

In [25]:
claims['Reimbursement Amount'] = np.where( claims['Claim Status'] == 'Approved', 
                                           claims['Billing Amount'] * np.random.uniform(0.80, 0.95, len(claims)), 
                                           np.where( claims['Claim Status'] == 'Pending', 
                                                     claims['Billing Amount'] * np.random.uniform(0.40, 0.70, len(claims)), 0 ) )

In [26]:
claims['Reimbursement Amount'] = claims['Reimbursement Amount'].round(2)

##### Patient Payable

In [27]:
claims['Patient Payable'] = ( claims['Billing Amount'] - claims['Reimbursement Amount'] ).round(2)

In [28]:
claims.head()

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,...,Discharge Date,Medication,Test Results,Claim ID,Length of Stay,Age Group,Cost Category,Claim Status,Reimbursement Amount,Patient Payable
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,...,2024-02-02,Paracetamol,Normal,CLM000001,2,19-35,Medium,Approved,16588.82,2267.46
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,...,2019-08-26,Ibuprofen,Inconclusive,CLM000002,6,51-65,Medium,Pending,19478.92,14164.41
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,...,2022-10-07,Aspirin,Normal,CLM000003,15,65+,Medium,Approved,25696.08,2259.02
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,...,2020-12-18,Ibuprofen,Abnormal,CLM000004,30,19-35,High,Approved,30650.29,7259.49
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,...,2022-10-09,Penicillin,Abnormal,CLM000005,20,36-50,Low,Approved,11410.65,2827.67


In [29]:
claims.shape

(54860, 22)

In [30]:
claims.columns

Index(['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition',
       'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider',
       'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date',
       'Medication', 'Test Results', 'Claim ID', 'Length of Stay', 'Age Group',
       'Cost Category', 'Claim Status', 'Reimbursement Amount',
       'Patient Payable'],
      dtype='object')

In [31]:
claims[['Billing Amount', 'Reimbursement Amount', 'Patient Payable']].describe()

,Billing Amount,Reimbursement Amount,Patient Payable
count,54860.000000,54860.000000,54860.000000
mean,25594.633637,18922.141051,6672.492561
std,14175.867041,13513.519333,9224.667476
min,9.238787,0.000000,0.670000
25%,13299.747940,6792.362500,1757.912500
50%,25593.873000,18204.805000,3515.270000
75%,37847.066671,30212.170000,6590.847500
max,52764.276736,49016.940000,52373.030000


##### Saving the Dataset

In [33]:
claims.to_csv('datasets/healthcare_claims_clean.csv', index=False)